# Timerseries Analysis

In [ ]:
import librosa
import numpy as np
import pandas as pd
from tslearn.shapelets import LearningShapelets
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import glob

## Load Audio and Extract Features

Load each file, trim, normalize amplitude, beattrack and then extract features.

In [2]:
def extract_beat_sync_features(
    mp3_path: str,
    sr: int = 22050,
    trim_db: int = 25,
    hop_length: int = 512,
    n_mfcc: int = 13,
) -> dict:
    y, sr = librosa.load(mp3_path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=trim_db)

    # normalize peak amplitude
    peak = np.max(np.abs(y)) + 1e-9
    y = y / peak

    duration = librosa.get_duration(y=y, sr=sr)

    # beat tracking
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    tempo, beat_frames = librosa.beat.beat_track(
        onset_envelope=onset_env, sr=sr, hop_length=hop_length
    )
    if len(beat_frames) < 2:
        # fallback: if beat tracking fails, treat it like one "beat" at start
        beat_frames = np.array([0], dtype=int)

    beat_times = librosa.frames_to_time(
        beat_frames, sr=sr, hop_length=hop_length
    )

    # frame-level features
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length
    )  # (n_mfcc, n_frames)

    chroma = librosa.feature.chroma_stft(
        y=y, sr=sr, hop_length=hop_length
    )  # (12, n_frames)

    # use the same onset envelope as a 1D rhythmic/dynamic feature
    onset = onset_env.reshape(1, -1)  # (1, n_frames)

    # beat-synchronous aggregation (median is robust)
    mfcc_bs = librosa.util.sync(mfcc, beat_frames, aggregate=np.median).T
    chroma_bs = librosa.util.sync(chroma, beat_frames, aggregate=np.median).T
    onset_bs = librosa.util.sync(onset, beat_frames, aggregate=np.median).T

    return {
        "mfcc_bs": mfcc_bs,
        "chroma_bs": chroma_bs,
        "onset_bs": onset_bs,
        "tempo": float(tempo),
        "beat_times": beat_times,
        "duration": float(duration),
    }

features = []
for mp3_path in sorted(glob.glob("../dataset/fedez_fibra/*.mp3"))[:10]:
    out = extract_beat_sync_features(mp3_path)
    out["artist"] = mp3_path.split(" ")[0].split("/")[-1]
    features.append(out)

pd.DataFrame(features).head()

/var/folders/xt/8hcc3d6j7l565z9k6z0w88jh0000gn/T/ipykernel_6244/4106808158.py:51: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "tempo": float(tempo),


,mfcc_bs,chroma_bs,onset_bs,tempo,beat_times,duration,artist
0,"[[-350.313, 91.51345, 13.632149, 17.825752, 10...","[[0.438727, 0.38372946, 0.24992833, 0.44548976...","[[0.3700195], [0.50319743], [0.61256325], [0.6...",89.102909,"[1.0913378684807256, 1.787936507936508, 2.4613...",119.675646,ART07024718
1,"[[-242.33176, 70.255615, 21.659698, -4.751523,...","[[0.22477466, 0.41218686, 0.20322362, 0.043952...","[[1.5244005], [1.012903], [1.3447756], [1.2036...",123.046875,"[2.507755102040816, 2.995374149659864, 3.50621...",122.694240,ART07024718
2,"[[-59.30338, 49.993156, 17.60097, 12.012684, -...","[[0.33985, 0.34022728, 0.3631826, 0.87398976, ...","[[0.0], [0.84293026], [0.9664582], [1.4517415]...",123.046875,"[0.06965986394557823, 0.5572789115646258, 1.04...",185.527438,ART07024718
3,"[[-52.30634, 85.76688, -3.730764, 64.843124, 3...","[[1.0, 0.8255083, 0.7445236, 0.6228194, 0.4834...","[[0.0], [0.31376678], [0.284397], [0.33596697]...",92.285156,"[0.06965986394557823, 0.8591383219954648, 1.50...",181.998005,ART07024718
4,"[[-142.6217, 22.849413, -68.8163, 13.575657, -...","[[0.3422467, 0.3433115, 0.48477343, 0.82623625...","[[0.0], [0.5330403], [0.54297143], [0.29903427...",129.199219,"[0.06965986394557823, 0.5340589569160997, 0.99...",196.742676,ART07024718


## Feature Aggregation for Clustering

We convert each mp3 into a single fixed-length feature vector by first computing
beat-synchronous audio descriptors and then aggregating them across
the whole track. This gives a tabular dataset suitable for standard clustering
(without DTW or subsequences).

Selected aggregated features per song:

- Basic track/tempo descriptors
  - duration: total length in seconds
  - tempo: estimated BPM from beat tracking
  - n_beats: number of detected beats (rough proxy for song length / beat density)

- MFCC statistics (timbre / production / "color")
  - mfcc_mean_*: mean of each MFCC coefficient across beats
  - mfcc_std_*: standard deviation of each MFCC coefficient across beats
  - mfcc_var_mean: average variance across MFCC coefficients (overall timbre variability)

- Chroma statistics (tonality / harmonic content distribution)
  - chroma_mean_*: average energy for each pitch class (12 dimensions)
  - chroma_std_*: variability of each pitch class across beats
  - chroma_entropy: entropy of the normalized chroma mean vector
    (higher = more evenly spread pitch classes; lower = more dominated by a few)
  - chroma_max_frac: fraction of energy in the most dominant pitch class

- Onset strength statistics (rhythmic density / dynamic “punch”)
  - onset_mean: average and variability of beat-synchronous onset strength
  - onset_std: average and variability of beat-synchronous onset strength
  - onset_p95: 95th percentile (captures strong accent peaks)
  - onset_peakiness: ratio p95 / mean (how “spiky” the accents are)
  - onset_diff_mean: mean / 95th percentile of absolute beat-to-beat
    changes in onset strength (proxy for structural/dynamic changes)

These features intentionally summarize timbre (MFCC), harmony (chroma) and
rhythm/dynamics (onset) in a compact way, enabling clustering on aggregated
metadata rather than raw time series alignment.

In [ ]:
def summarize_song_features(out: dict) -> dict:
    mfcc = out["mfcc_bs"]          # (T, n_mfcc)
    chroma = out["chroma_bs"]      # (T, 12)
    onset = out["onset_bs"][:, 0]  # (T,)

    eps = 1e-9

    agg = {}
    agg["artist"] = out["artist"]
    agg["tempo"] = out["tempo"]
    agg["duration"] = out["duration"]
    agg["n_beats"] = float(mfcc.shape[0])

    # mean and std per coefficient
    mfcc_mean = mfcc.mean(axis=0)
    mfcc_std = mfcc.std(axis=0)
    for i, v in enumerate(mfcc_mean, start=1):
        agg[f"mfcc_mean_{i:02d}"] = float(v)
    for i, v in enumerate(mfcc_std, start=1):
        agg[f"mfcc_std_{i:02d}"] = float(v)

    agg["mfcc_var_mean"] = float(np.mean(mfcc_std**2))

    chroma_mean = chroma.mean(axis=0)
    chroma_std = chroma.std(axis=0)

    # normalize chroma_mean to a distribution for entropy-like features
    p = chroma_mean / (chroma_mean.sum() + eps)

    for k, v in enumerate(chroma_mean):
        agg[f"chroma_mean_{k:02d}"] = float(v)
    for k, v in enumerate(chroma_std):
        agg[f"chroma_std_{k:02d}"] = float(v)

    agg["chroma_entropy"] = float(-(p * np.log(p + eps)).sum())
    agg["chroma_max_frac"] = float(p.max())  # dominance of one pitch class

    # Onset strength summary stats (rhythm / dynamics proxy)
    agg["onset_mean"] = float(onset.mean())
    agg["onset_std"] = float(onset.std())
    agg["onset_p95"] = float(np.percentile(onset, 95))
    agg["onset_peakiness"] = float(
        np.percentile(onset, 95) / (onset.mean() + eps)
    )

    # simple "section change" proxy: how often onset changes sharply beat-to-beat
    diff = np.abs(np.diff(onset))
    agg["onset_diff_mean"] = float(diff.mean()) if diff.size else 0.0
    agg["onset_diff_p95"] = float(np.percentile(diff, 95)) if diff.size else 0.0

    return agg


# extract aggregated metadata features for all songs
agg_rows = []
for out in features:
    agg_rows.append(summarize_song_features(out))

df = pd.DataFrame(agg_rows)
X = df.drop(columns=["artist"])

print(df.shape)
pd.set_option('display.max_columns', None)
df.head()

(10, 63)


,artist,tempo,duration,n_beats,mfcc_mean_01,mfcc_mean_02,mfcc_mean_03,mfcc_mean_04,mfcc_mean_05,mfcc_mean_06,mfcc_mean_07,mfcc_mean_08,mfcc_mean_09,mfcc_mean_10,mfcc_mean_11,mfcc_mean_12,mfcc_mean_13,mfcc_std_01,mfcc_std_02,mfcc_std_03,mfcc_std_04,mfcc_std_05,mfcc_std_06,mfcc_std_07,mfcc_std_08,mfcc_std_09,mfcc_std_10,mfcc_std_11,mfcc_std_12,mfcc_std_13,mfcc_var_mean,chroma_mean_00,chroma_mean_01,chroma_mean_02,chroma_mean_03,chroma_mean_04,chroma_mean_05,chroma_mean_06,chroma_mean_07,chroma_mean_08,chroma_mean_09,chroma_mean_10,chroma_mean_11,chroma_std_00,chroma_std_01,chroma_std_02,chroma_std_03,chroma_std_04,chroma_std_05,chroma_std_06,chroma_std_07,chroma_std_08,chroma_std_09,chroma_std_10,chroma_std_11,chroma_entropy,chroma_max_frac,onset_mean,onset_std,onset_p95,onset_peakiness,onset_diff_mean,onset_diff_p95
0,ART07024718,89.102909,119.675646,175.0,-158.168900,103.729576,3.666085,38.286690,6.462377,17.200384,-9.020933,8.186444,-15.491857,12.498502,-3.951241,-0.101257,-5.834316,106.947578,20.744963,15.191677,13.263687,10.978548,8.240155,9.611194,7.740864,9.499740,7.823032,5.332959,6.590290,5.685448,990.093628,0.222746,0.205942,0.216175,0.316212,0.504544,0.347253,0.316143,0.431757,0.379747,0.411046,0.306597,0.272804,0.149090,0.148490,0.129447,0.162394,0.253724,0.173641,0.168443,0.240155,0.171074,0.245911,0.137189,0.132367,2.448570,0.128351,1.026717,0.360128,1.549208,1.508895,0.209277,0.556716
1,ART07024718,123.046875,122.694240,243.0,-51.307961,94.274467,-10.893713,10.078119,5.699718,5.601115,-5.013778,3.204492,-5.247590,1.615436,-2.062113,1.850235,1.923126,42.064415,18.266651,14.100110,13.038759,13.833110,9.725368,8.494978,8.429314,6.369573,6.134578,5.792819,5.792889,6.600730,237.688614,0.388347,0.472805,0.524635,0.471819,0.435879,0.364299,0.319499,0.312314,0.373174,0.363408,0.398952,0.410998,0.142900,0.192324,0.205464,0.179194,0.179943,0.163741,0.153971,0.150786,0.196918,0.181945,0.191594,0.165256,2.473500,0.108482,1.307706,0.356755,1.850940,1.415410,0.290918,0.727805
2,ART07024718,123.046875,185.527438,364.0,-63.896343,82.414986,-1.491513,8.552963,6.751056,5.475074,-5.372694,2.871174,-5.015740,2.478053,-4.113449,1.432435,-3.011876,95.649117,24.659386,22.954193,13.523681,15.784090,11.420725,9.411446,11.458076,8.550922,7.960243,7.538289,7.624220,9.029694,876.848572,0.405003,0.417858,0.417483,0.429877,0.455901,0.386755,0.320775,0.366050,0.387760,0.457095,0.441335,0.409417,0.196269,0.200238,0.206738,0.205456,0.215875,0.181878,0.158671,0.181558,0.191437,0.258999,0.230848,0.194804,2.480602,0.093374,1.067506,0.340703,1.563847,1.464953,0.230953,0.711033
3,ART07024718,92.285156,181.998005,251.0,-12.326166,89.520187,-3.961415,30.572599,11.225276,11.424213,-2.725905,7.167484,-1.636661,6.239148,-2.517207,5.088517,-0.525433,51.730137,16.901812,12.708161,10.817780,9.264805,6.702595,6.865829,6.242379,6.102638,6.775903,5.746853,5.209935,4.239802,278.336060,0.360818,0.319289,0.433231,0.307337,0.299752,0.390453,0.307364,0.303901,0.290493,0.399368,0.518921,0.349244,0.228451,0.161131,0.246348,0.170704,0.151589,0.235229,0.164671,0.196473,0.157645,0.191881,0.287599,0.167687,2.468838,0.121238,0.928150,0.206752,1.219235,1.313618,0.156321,0.425156
4,ART07024718,129.199219,196.742676,356.0,-30.351585,69.020233,-4.350806,23.973255,8.916329,-4.661334,-3.560339,3.058527,-2.507047,4.128809,-1.376578,4.342422,1.433885,51.285381,16.148561,23.267908,12.147522,10.354442,14.248526,8.970078,6.350777,8.648898,7.430833,6.105474,7.238554,5.827940,328.047150,0.400180,0.407072,0.418192,0.521001,0.374701,0.316217,0.409472,0.345433,0.392402,0.342894,0.434325,0.523048,0.153682,0.201147,0.165594,0.226660,0.179769,0.151891,0.231444,0.162972,0.204666,0.161959,0.196356,0.249653,2.473906,0.107074,1.095254,0.333482,1.683079,1.536702,0.246248,0.824179


## Impute Missing Values

In [4]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

## Scale Features

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

(10, 62)